In [3]:
#InterQuantileRange(IQR) quarters method   removes outliers that deviate from majority
import polars as pl

df = pl.DataFrame({
    "salary":[4000, 4200, 4300, 4400, 4500, 4600, 4700, 4800, 100000, 110000]
})
df

salary
i64
4000
4200
4300
4400
4500
4600
4700
4800
100000


In [24]:
q1 = df.select(pl.col('salary').quantile(0.25))
q1 = q1.item()

In [25]:
q3 = df.select(pl.col('salary').quantile(0.75)).item()

iqr = q3 - q1
q3

4800.0

In [26]:
iqr

500.0

In [27]:
lower_bound = q1 - (1.5 * iqr)
upper_bound = q3 + (1.5 * iqr)

In [28]:
lower_bound

3550.0

In [29]:
filtered = df.filter(
    (pl.col('salary') >= lower_bound) & (pl.col('salary') <= upper_bound)
)
filtered

salary
i64
4000
4200
4300
4400
4500
4600
4700
4800


In [30]:
import polars as pl
from sklearn.preprocessing  import LabelEncoder

In [38]:
df = pl.DataFrame({
    "fruit": ['apple', 'banana', 'apple', 'orange', 'banana']
})
df

fruit
str
"""apple"""
"""banana"""
"""apple"""
"""orange"""
"""banana"""


In [39]:
df['fruit'].to_numpy()

array(['apple', 'banana', 'apple', 'orange', 'banana'], dtype=object)

In [32]:
le = LabelEncoder()
encoded = le.fit_transform(df['fruit'].to_numpy())
encoded

array([0, 1, 0, 2, 1])

In [33]:
f_encoded = df.with_columns(pl.Series('fruit_encoded', encoded))

In [34]:
f_encoded

fruit,fruit_encoded
str,i64
"""apple""",0
"""banana""",1
"""apple""",0
"""orange""",2
"""banana""",1


In [40]:
from sklearn.preprocessing  import OneHotEncoder

data = {
    "name": ["Alice", "Bob", "Charlie", "Diana", "Ethan"],
    "city": ["New York", "London", "Tokyo", "Paris", "Berlin"],
}

df = pl.DataFrame(data)
df

name,city
str,str
"""Alice""","""New York"""
"""Bob""","""London"""
"""Charlie""","""Tokyo"""
"""Diana""","""Paris"""
"""Ethan""","""Berlin"""


In [42]:
cities = df.select('city').to_numpy()

In [43]:
ohe = OneHotEncoder(sparse_output=False)

city_encoded = ohe.fit_transform(cities)
city_encoded

array([[0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0.]])

In [44]:
encoded_cols = ohe.get_feature_names_out(['city'])

In [46]:
encoded_df = pl.DataFrame({col: city_encoded[:, i] for i, col in enumerate(encoded_cols)})

In [47]:
n_df = df.drop('city').hstack(encoded_df)
n_df

name,city_Berlin,city_London,city_New York,city_Paris,city_Tokyo
str,f64,f64,f64,f64,f64
"""Alice""",0.0,0.0,1.0,0.0,0.0
"""Bob""",0.0,1.0,0.0,0.0,0.0
"""Charlie""",0.0,0.0,0.0,0.0,1.0
"""Diana""",0.0,0.0,0.0,1.0,0.0
"""Ethan""",1.0,0.0,0.0,0.0,0.0
